# Mini Workshop — ONNX

> *PyTorch is the language you wrote your model in. ONNX is the language your model travels in.*

In this Mini, we'll:
1. **Run** SceneSeg (Autoware perception) **as PyTorch** — the prototyping path
2. **Export** it the way production AV teams do — bake the preprocessing and post-processing into a single ONNX file
3. **Run** your ONNX directly on raw images and verify the segmentation matches PyTorch

The big Workshop takes the same export and pushes it through TensorRT FP16 + INT8.


## Setup


In [ ]:
!pip install onnxruntime-gpu gdown onnxscript onnx -q


In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
import numpy as np
import cv2
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import onnxruntime as ort
import onnx
import glob, os

print(f"PyTorch {torch.__version__} | ORT {ort.__version__} | ONNX {onnx.__version__}")


## Download — SceneSeg model & Waymo frames


In [ ]:
!mkdir -p /content/data /content/models

# SceneSeg PyTorch (TorchScript-traced)
!gdown '1G2pKrjEGLGY1ouQdNPh11N-5LlmDI7ES' -O /content/models/SceneSeg_traced.pt

# Waymo driving frames
!wget -qq https://optical-flow-data.s3.eu-west-3.amazonaws.com/waymo_images.zip -O /content/data/waymo.zip
!unzip -qq /content/data/waymo.zip -d /content/data/

print(f"\nModel: {os.path.getsize('/content/models/SceneSeg_traced.pt')/1e6:.1f} MB")


In [ ]:
# SceneSeg expects 320 x 640 input (confirm in Netron, or by inspecting the ONNX after export).
H, W = 320, 640


## Load a Waymo frame


In [ ]:
frames = sorted(glob.glob('/content/data/night/front_images_night/*.jpg'))
frame_path = frames[100]

frame_bgr = cv2.imread(frame_path)
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

# Standard preprocessing (resize + ToTensor + ImageNet normalize)
preprocess = T.Compose([
    T.Resize((H, W)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
x = preprocess(Image.fromarray(frame_rgb)).unsqueeze(0)   # (1,3,H,W) normalized

# We'll also need the [0,1] version (no normalization) for the production-style ONNX in Part 3.
x_01 = T.Compose([T.Resize((H, W)), T.ToTensor()])(Image.fromarray(frame_rgb)).unsqueeze(0)
x_01_np = x_01.numpy()

plt.imshow(cv2.resize(frame_rgb, (W, H))); plt.axis('off')
plt.title('Input: Waymo driving frame'); plt.show()


In [ ]:
COLORS = np.array([
    [240,  40,  40],   # 0 background
    [180,  60, 200],   # 1 foreground (cars / pedestrians)
    [ 80, 200,  80],   # 2 drivable road
], dtype=np.uint8)


---
## Part 1 — Run as PyTorch

Standard prototyping path: load the model, preprocess in Python, run. This is fine for research; the problem starts when you have to ship it.


In [ ]:
model = torch.jit.load('/content/models/SceneSeg_traced.pt', map_location='cpu')
model.eval()

with torch.no_grad():
    pt_logits = model(x)
    pt_map = pt_logits.argmax(dim=1)[0].numpy()

print(f"PyTorch output shape: {tuple(pt_logits.shape)}")

input_img = cv2.resize(frame_rgb, (W, H))
overlay   = cv2.addWeighted(input_img, 0.5, COLORS[pt_map], 0.5, 0)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].imshow(input_img);     axes[0].set_title('Input frame');           axes[0].axis('off')
axes[1].imshow(COLORS[pt_map]); axes[1].set_title('PyTorch — segmentation'); axes[1].axis('off')
axes[2].imshow(overlay);       axes[2].set_title('PyTorch — overlay');     axes[2].axis('off')
plt.tight_layout(); plt.show()


---
## Part 2 — The production move: bake everything into one ONNX

Here's what most tutorials teach: `torch.onnx.export(model, dummy, 'model.onnx')`. That gives you an ONNX file that takes the *same preprocessed tensor* as PyTorch — meaning your deployment code (C++, ROS node, Triton server, ...) has to re-implement the preprocessing pipeline in its own language. Three common ways that goes wrong:

- **Mean/std mismatch.** Someone reads `[0.485, 0.456, 0.406]` from one repo and `[0.5, 0.5, 0.5]` from another. Output looks "almost right" but hallucinates lane lines.
- **BGR vs RGB.** OpenCV is BGR, PyTorch is RGB. One missed conversion and the model fails on red traffic lights.
- **Argmax in two languages.** Logits come out of the model; converting to a class map needs to happen somewhere. Two implementations = two places to be wrong.

Production teams ship **one ONNX file that does everything**: takes a raw image, returns the final segmentation map. Normalization and argmax baked in.

We do that with a **wrapper module**:


In [ ]:
class ExportableSceneSeg(nn.Module):
    """Production wrapper: takes a [0,1] float image, returns class indices directly.

    The mean/std and argmax are baked into the graph. Deployment code never has to
    know what normalization this model was trained with — it just feeds raw pixels.
    """
    def __init__(self, base: nn.Module):
        super().__init__()
        self.base = base
        # Buffers travel with the model — saved into the ONNX as constants.
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, image_01: torch.Tensor) -> torch.Tensor:
        # image_01: (1, 3, H, W) in [0, 1]
        x = (image_01 - self.mean) / self.std
        logits = self.base(x)
        return logits.argmax(dim=1).to(torch.uint8)   # (1, H, W) class indices


### A note on `dynamo=True`

Since PyTorch 2.5, `torch.onnx.export(..., dynamo=True)` is the **recommended** path. It uses `torch.export` + Torch FX for cleaner graph capture, supports dynamic shapes natively, and handles control flow better than the legacy tracer.

**We use `dynamo=False`** here because Autoware ships SceneSeg as a TorchScript-traced `.pt`, and the dynamo exporter doesn't accept `ScriptModule` inputs yet. For models you write from scratch (regular `nn.Module`), use `dynamo=True`.

This is the kind of compatibility decision you'll make over and over in real AV stacks: "the modern way is X, but my dependency is Y, so I do Z."


In [ ]:
# TODO 1 — Sanity-check the wrapper before exporting:
#   pass x_01 through `exportable` (you'll need to instantiate it from `model`),
#   then verify it matches `pt_map` from Part 1.
exportable = ...
# ...

# TODO 2 — Build a dummy input with the shape the wrapper expects.
# Hint: (1, 3, H, W). The wrapper takes [0,1] floats, so all-zeros is fine for tracing.
dummy = ...

# TODO 3 — Call torch.onnx.export with:
#   - exportable (NOT the raw model — we want preprocessing baked in)
#   - output path '/content/models/SceneSeg_PROD.onnx'
#   - opset_version=17
#   - input_names=['image'], output_names=['segmentation']
#   - dynamo=False  (see note above on why)
torch.onnx.export(...)

# TODO 4 — Validate the file with onnx.checker.check_model and report file size.


---
## Part 3 — Run YOUR ONNX

Same input frame — but now the deployment side is dramatically simpler. **No normalization, no argmax, no Python helpers**. The runtime takes a raw image and returns the segmentation map.

This is what gets handed to the C++/ROS/Triton team in production. They never have to know the model's mean/std.


In [ ]:
# TODO 5 — Build an InferenceSession on '/content/models/SceneSeg_PROD.onnx'.
# Bonus: pass a SessionOptions() with graph_optimization_level = ORT_ENABLE_ALL.
sess = ...

# TODO 6 — Run inference on x_01_np (input name: 'image'). Output shape is (1, H, W).
ort_map = ...

# TODO 7 — Visualize: input | PyTorch overlay | your ONNX overlay (3 panels).

# TODO 8 — Compute pixel agreement (pt_map == ort_map).mean() * 100 and decide pass/fail.


---
## Bonus — What's inside the file?

ONNX files are just graphs: nodes (operators) connected by tensors. Let's open ours and see what got baked in.


In [ ]:
m = onnx.load('/content/models/SceneSeg_PROD.onnx')

op_counts = {}
for node in m.graph.node:
    op_counts[node.op_type] = op_counts.get(node.op_type, 0) + 1

total_ops  = sum(op_counts.values())
unique_ops = len(op_counts)
top_5      = sorted(op_counts.items(), key=lambda kv: -kv[1])[:5]

print(f"Total nodes:        {total_ops}")
print(f"Unique op types:    {unique_ops}")
print(f"Inputs / outputs:   {len(m.graph.input)} / {len(m.graph.output)}")
print(f"\nTop 5 operators:")
for op, n in top_5:
    print(f"  {op:<24} {n}")

# Confirm normalization and argmax made it into the graph (not Python helpers anymore)
print(f"\nSub  (normalization step):  {op_counts.get('Sub', 0)}  (expect ≥ 1)")
print(f"Div  (normalization step):  {op_counts.get('Div', 0)}  (expect ≥ 1)")
print(f"ArgMax (post-processing):   {op_counts.get('ArgMax', 0)}  (expect 1)")
print(f"\nTip: open SceneSeg_PROD.onnx in https://netron.app for a visual graph.")


---
## 🎯 Your Turn — Export your Module 1 models to ONNX

In earlier modules, you built three optimized DeepLab variants:
- A **pruned** model (Pruning module)
- A **statically quantized** model (Quantization module)
- A **distilled** student model (Knowledge Distillation module)

All three are real wins — **in PyTorch**. To ship any of them, they need to leave PyTorch. That means ONNX, using exactly the pattern you just learned.

### The challenge

Pick **at least one** of the three (the distilled student is the easiest place to start) and:

1. **Wrap it** in an `ExportableDeepLab(nn.Module)` that bakes in the normalization and argmax — same pattern as `ExportableSceneSeg` above.
2. **Export** to ONNX with `torch.onnx.export(...)`.
3. **Verify** by running with `onnxruntime` and computing pixel agreement vs the PyTorch original. Aim for >99%.
4. **Visualize** input | PyTorch overlay | your ONNX overlay on a sample Cityscapes frame.

### Stretch — export all three

Compare the file sizes of the resulting `.onnx` files. Which optimization produced the smallest deployable artifact? Is it the same as the smallest **PyTorch** model? Explain what you observe.

### Hints

- **Pruned models** export trivially — pruning sets weights to zero; the graph structure is unchanged.
- **Distilled students** export trivially — they're smaller `nn.Module`s, nothing special.
- **Statically quantized models** are trickier. PyTorch's `torch.quantization` workflow produces models with `QuantStub`/`DeQuantStub` modules and `quint8` tensors. Exporting needs `opset_version >= 13`, and the result uses the QDQ (Quantize–DeQuantize) format. If you hit walls here — that's expected, and an honest preview of why the Workshop quantizes at **deployment time** with TensorRT instead of in PyTorch.

### What to submit / share

Drop your three `.onnx` files (or one + notes) and a short write-up of the file-size comparison. If something didn't export, write up *why* — the failure mode is the lesson.


---
## What's next

You exported a production-style ONNX file: it takes a raw image and returns a segmentation map directly. Normalization and argmax are baked into the graph — your C++/ROS/Triton deployment never has to know what mean/std this model trained with.

In the Workshop (`Model_Deployment.ipynb`), we take this ONNX further:
1. Compile to a **TensorRT engine** for the GPU
2. Calibrate to **INT8** with real driving frames
3. Benchmark all 5 runtimes side by side, render a comparison video

See you there.
